# Educación preescolar, básica y media por municipio (MinEducación)

Indicadores oficiales del sector educativo para cada municipio y año: cobertura neta y bruta, deserción, aprobación, reprobación, repitencia, tamaño promedio de los grupos y sedes conectadas a internet. Es una base **panel** (municipio × año), pequeña y completa, ideal para empezar.

|  |  |
|---|---|
| **Entidad** | Ministerio de Educación Nacional (MEN) |
| **Conjunto** | Estadísticas en educación preescolar, básica y media por municipio · [`nudc-7mev`](https://www.datos.gov.co/d/nudc-7mev) |
| **Una fila es** | un municipio en un año |
| **Cobertura** | los 1.122 municipios y áreas no municipalizadas |
| **Periodo** | 2011 a 2024 |
| **Tamaño total** | 15.707 filas y 41 columnas (consultado el 31/10/2026) |
| **Actualización** | anual; última carga el 13/11/2025 |
| **Requisitos** | ninguno (token de datos.gov.co opcional) |
| **Tiempo de descarga** | menos de 1 minuto para todo el país |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base

Las tasas están en porcentaje (0 a 100). Las de deserción, aprobación, reprobación y repitencia se refieren al **sector oficial**.

| Columna | Qué es | Tipo |
|---|---|---|
| `anio` | Año del indicador | número |
| `cod_mpio`, `municipio` | Código DANE (5 dígitos) y nombre del municipio | texto |
| `cod_dpto`, `departamento` | Código y nombre del departamento | texto |
| `cod_etc`, `etc` | Entidad territorial certificada que administra la educación | texto |
| `poblacion_5_16` | Población de 5 a 16 años (proyecciones DANE) | número |
| `tasa_matriculacion_5_16` | Porcentaje de la población de 5 a 16 años que asiste | número |
| `cobertura_neta`, `cobertura_neta_<nivel>` | Matriculados en la edad teórica / población en esa edad (transición, primaria, secundaria, media) | número |
| `cobertura_bruta`, `cobertura_bruta_<nivel>` | Todos los matriculados / población en edad teórica (puede pasar de 100) | número |
| `tamano_promedio_de_grupo` | Estudiantes por grupo | número |
| `sedes_conectadas_a_internet` | Porcentaje de sedes oficiales con internet | número |
| `desercion`, `desercion_<nivel>` | Porcentaje de matriculados que abandonan durante el año | número |
| `aprobacion`, `reprobacion`, `repitencia` y sus niveles | Porcentaje que aprueba, reprueba o repite el grado | número |

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Regresión | `desercion` del municipio | cobertura, tamaño de grupo, internet, población; y otras bases unidas por `cod_mpio` y `anio` |
| Clasificación | ¿El municipio queda en el 25 % con mayor deserción? | las mismas |
| Pronóstico | `desercion` del año siguiente | valores del municipio en años anteriores (rezagos) |
| Agrupamiento | tipologías de municipios por su perfil educativo | coberturas y tasas escaladas |

**Cuidados antes de modelar**

- **Fuga de información:** aprobación + reprobación + deserción suman cerca de 100. Si predice una, no use las otras dos del mismo año.
- Es un panel: si parte al azar, el mismo municipio queda en entrenamiento y prueba. Valide **por año** (entrene con años anteriores) o por municipio.
- Las coberturas pueden superar 100 % (estudiantes de otras edades o que viven en otro municipio): no es un error.
- Entre 2018 y 2019 cambiaron las proyecciones de población (censo 2018): las coberturas pueden saltar entre esos años.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

La base es pequeña: por defecto se descarga todo el país y se filtra después.

> **Token opcional.** datos.gov.co funciona sin clave, pero con un *app token* gratuito las descargas grandes fallan menos: cree una cuenta en datos.gov.co → *Developer Settings* → *Create New App Token*, y antes de ejecutar escriba `os.environ["SOCRATA_APP_TOKEN"] = "su_token"`.

In [ ]:
DEPARTAMENTO = None    # p. ej. "Boyacá", con o sin tildes. None = todo el país
ANIO_INI     = 2011    # primer año (desde 2011)
ANIO_FIN     = 2024    # último año (hasta 2024)
INDICADORES  = None    # None = todas las columnas; o una lista, p. ej. ["cobertura_neta", "desercion", "aprobacion", "repitencia"]

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

# ---------- datos.gov.co (API Socrata, consultas en lenguaje SoQL) ----------
BASE = "https://www.datos.gov.co/resource/{}.json"
TOKEN = os.getenv("SOCRATA_APP_TOKEN")      # opcional: os.environ["SOCRATA_APP_TOKEN"] = "su_token"
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}

def socrata(dataset_id, select=None, where=None, group=None, order=None,
            max_rows=None, limit=50_000, timeout=300, reintentos=3, mensajes=True):
    """Descarga un conjunto de datos.gov.co por páginas, con reintentos. Devuelve un DataFrame."""
    if max_rows: limit = min(limit, max_rows)
    params = {"$limit": limit}
    if select: params["$select"] = select
    if where:  params["$where"]  = where
    if group:  params["$group"]  = group
    params["$order"] = order or (group if group else ":id")    # orden estable para paginar
    partes, offset = [], 0
    while True:
        params["$offset"] = offset
        for intento in range(reintentos):
            try:
                r = requests.get(BASE.format(dataset_id), params=params, headers=HEADERS, timeout=timeout)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if intento == reintentos - 1:
                    raise
                print(f"  reintento {intento + 1}: {e}")
                time.sleep(5 * (intento + 1))
        lote = r.json()
        if not lote:
            break
        partes.append(pd.DataFrame(lote))
        offset += len(lote)
        if mensajes:
            print(f"  {dataset_id}: {entero(offset)} filas descargadas", end="\r")
        if len(lote) < limit or (max_rows and offset >= max_rows):
            break
        time.sleep(0.3)
    if mensajes:
        print()
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    return df.head(max_rows) if max_rows else df

def contar(dataset_id, where=None):
    """Cuántas filas cumplen el filtro, sin descargarlas."""
    r = socrata(dataset_id, select="count(*) as n", where=where, order="n", mensajes=False)
    return int(r["n"].iloc[0]) if len(r) else 0

def lista_soql(valores):
    """['a', "b'c"] -> ('a', 'b''c') para usar con IN en SoQL."""
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

def canon(texto):
    """Nombre de departamento comparable entre fuentes (con o sin tildes, 'Bogotá D.C.', 'San Andrés…')."""
    t = sin_tildes(texto)
    if "BOGOTA" in t: return "BOGOTA"
    if "SAN ANDRES" in t: return "SAN ANDRES"
    return t.replace("DEPARTAMENTO DE ", "")

def valores_de(dataset_id, campo, texto, where=None, contiene=False):
    """Busca cómo está escrito `texto` en la columna `campo` (cada fuente lo escribe distinto)."""
    df = socrata(dataset_id, select=campo, where=where, group=campo, mensajes=False)
    if df.empty:
        return []
    todos = df[campo].dropna().unique()
    if contiene:
        hallados = [v for v in todos if sin_tildes(texto) in sin_tildes(v)]
    else:
        hallados = [v for v in todos if canon(v) == canon(texto)]
    if not hallados:
        raise ValueError(f"No encontré '{texto}' en {campo}. Valores posibles: {sorted(todos)}")
    return hallados

def cod5(s):
    """Código DANE de municipio a 5 dígitos como texto (algunas fuentes traen 8 o pierden el cero)."""
    s = pd.Series(s).astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return pd.Series(np.where(s.str.len() >= 7, s.str.zfill(8).str[:5], s.str.zfill(5)), index=s.index)

def a_numero(df, cols):
    for c in cols:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# Códigos DANE de los departamentos (para filtrar por código, que no depende de tildes)
COD_DPTO = {"ANTIOQUIA": "05", "ATLANTICO": "08", "BOGOTA": "11", "BOLIVAR": "13", "BOYACA": "15",
            "CALDAS": "17", "CAQUETA": "18", "CAUCA": "19", "CESAR": "20", "CORDOBA": "23",
            "CUNDINAMARCA": "25", "CHOCO": "27", "HUILA": "41", "LA GUAJIRA": "44", "MAGDALENA": "47",
            "META": "50", "NARINO": "52", "NORTE DE SANTANDER": "54", "QUINDIO": "63", "RISARALDA": "66",
            "SANTANDER": "68", "SUCRE": "70", "TOLIMA": "73", "VALLE DEL CAUCA": "76", "ARAUCA": "81",
            "CASANARE": "85", "PUTUMAYO": "86", "SAN ANDRES": "88", "AMAZONAS": "91", "GUAINIA": "94",
            "GUAVIARE": "95", "VAUPES": "97", "VICHADA": "99"}

def cod_dpto(nombre):
    """'Nariño' -> '52'."""
    c = canon(nombre)
    if c not in COD_DPTO:
        raise ValueError(f"No reconozco el departamento '{nombre}'. Use uno de: {sorted(COD_DPTO)}")
    return COD_DPTO[c]

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. Descarga

In [ ]:
EDU = "nudc-7mev"
print(f"Filas en la base: {entero(contar(EDU))}")
edu = socrata(EDU, where=f"a_o::number between {ANIO_INI} and {ANIO_FIN}")

## 5. Limpieza mínima

Los nombres de columna de datos.gov.co pierden las tildes (`deserci_n`); aquí se corrigen. Se quitan las filas del total nacional y se pasa todo a número.

In [ ]:
def limpiar_nombre(c):
    c = {"a_o": "anio", "c_digo_municipio": "cod_mpio", "c_digo_departamento": "cod_dpto", "c_digo_etc": "cod_etc"}.get(c, c)
    return c.replace("ci_n", "cion").replace("tama_o", "tamano")

edu = edu.rename(columns=limpiar_nombre)
edu = edu[edu["cod_mpio"].astype(str) != "0"].copy()              # filas del total nacional
edu["cod_mpio"] = cod5(edu["cod_mpio"])
edu["cod_dpto"] = edu["cod_mpio"].str[:2]
texto = ["cod_mpio", "municipio", "cod_dpto", "departamento", "cod_etc", "etc"]
edu = a_numero(edu, [c for c in edu.columns if c not in texto])

if DEPARTAMENTO:
    edu = edu[edu["cod_dpto"] == cod_dpto(DEPARTAMENTO)]
if INDICADORES:
    edu = edu[["anio"] + texto + list(INDICADORES)]
edu = edu.sort_values(["cod_mpio", "anio"]).reset_index(drop=True)
print(f"{entero(len(edu))} filas: {edu['cod_mpio'].nunique()} municipios × {edu['anio'].nunique()} años")

## 6. Revisión de la base

In [ ]:
ficha_rapida(edu)
edu.head()

*(Se completa al ejecutar.)*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.hist(edu["desercion"].dropna().clip(upper=15), bins=50, color=AZUL, edgecolor="white", linewidth=0.4)
a1.set_title("Deserción por municipio-año (%)"); a1.set_xlabel("deserción (valores mayores a 15 se muestran en 15)")
med = edu.groupby("anio")[["desercion", "cobertura_neta"]].median()
a2.plot(med.index, med["desercion"], color=NARANJA, marker="o", ms=4, label="deserción")
a2.set_title("Mediana de los municipios por año (%)"); a2.set_xlabel("año")
a3 = a2.twinx(); a3.plot(med.index, med["cobertura_neta"], color=AZUL, marker="o", ms=4); a3.grid(False)
a2.set_ylabel("deserción", color=NARANJA); a3.set_ylabel("cobertura neta", color=AZUL)
plt.tight_layout(); plt.show()
med.round(2).T

*(Se completa al ejecutar.)*

## 7. Guardar

In [ ]:
ruta = guardar(edu, etiqueta("educacion_municipal", DEPARTAMENTO or "pais"))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/educacion_municipal_pais.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.
5. Esta base se une con Saber 11 (nivel municipio), homicidios o agricultura por `cod_mpio` y `anio`; el cuaderno `15-panel-municipal` ya lo hace.